# OIM on PYNQ-Z2 - hardware measurements

**Every plot in this notebook is measured on the PL.** No simulated curve is
drawn anywhere. `oim_sim` is imported only for things that are not data:
graph definitions, the fixed-point quantisers, `stable_dt`, and
`brute_force_maxcut` for the true optimum a hardware result is scored against.

The one deliberate exception is **section 2**, which prints (does not plot) the
board's phase words beside the Python twin's. That is the correctness anchor -
the RTL is already proven bit-exact against the twin in simulation, so a
mismatch there means a wrong register value, not a wrong design. Delete it only
once you trust the build.

Build verified from `mainDesign.hwh`:
`oim_pynq` at **0x40000000**, range 4 KB, `C_S00_AXI_ADDR_WIDTH = 10`,
`NMAX = 16`, device `7z020clg400`, Vivado 2024.1.


## 1. Driver

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from pynq import Overlay, MMIO
plt.rcParams.update({"figure.dpi":110,"font.size":9,"axes.grid":True,"grid.alpha":.3})

TWO_PI = 2*np.pi
PW = 16
LSB_PER_RAD = (1 << PW)/TWO_PI
DEG_PER_LSB = 360.0/(1 << PW)


class OIMEngine:
    """AXI4-Lite driver for oim_pynq_top.

    Every quantiser below is the same arithmetic as oim_sim.py. If you change
    one, change both, or the bit-exactness check in section 2 will fail.
    """

    # ---- register map, byte offsets ----
    CTRL=0x000; STATUS=0x004; NSPINS=0x008; KC=0x00C; KS=0x010
    DTQ=0x014; NAMP=0x018; SEED=0x01C
    KS_START=0x020; KS_STOP=0x024; KS_INC=0x028; MAXSTEPS=0x02C
    TOL=0x030; STRIDE=0x034; HOLD=0x038; STEPCNT=0x03C
    DUTY=0x040; WINDOW=0x044; FW0=0x048; BASE=0x04C
    TOTAL=0x050; DFFQ=0x054; KSCUR=0x058; PHTAP=0x05C
    PHASE=0x080; FW=0x0C0; W=0x100; DW=0x140; JPACK=0x180
    XORC=0x200; EDGEC=0x240; TOGC=0x280; PHASEID=0x2C0
    DFF2Q=0x06C; DFFSQ=0x070; RSC=0x300; TOG2C=0x340; TOGSC=0x380   # v2 readout

    # ---- CTRL bits (0, 2 and 3 self-clear) ----
    B_START=1 << 0; B_SRST=1 << 1; B_RDSTART=1 << 2; B_LOADPH=1 << 3

    def __init__(self, bitfile="mainDesign.bit", ip="oim_pynq",
                 base=None, span=0x1000):
        self.ol = Overlay(bitfile)
        if base is None:
            base = self.ol.ip_dict[ip]["phys_addr"]
        self.mmio = MMIO(base, span)
        ident = self.rd(0x060)
        if ident not in (0x4F494D01, 0x4F494D02):
            raise RuntimeError(f"ID = {ident:#010x}, expected 0x4f494d02")
        self.version = ident & 0xFF      # 2 = v2 readout: RS-FF, 2nd D-FF, SHIL D-FF
        self.n = 6
        self.fw0, self.fw, self.phtap_q = 256, [256]*16, 0x1C71   # reset values
        self.reset()

    def wr(self, a, v):
        self.mmio.write(a, int(v) & 0xFFFFFFFF)

    def rd(self, a):
        return self.mmio.read(a)

    def reset(self):
        self.wr(self.CTRL, self.B_SRST)
        self.wr(self.CTRL, 0)

    # ---------- quantisation, identical to oim_sim.py ----------
    @staticmethod
    def q_kc(x):
        return int(round(x*4096))                       # Q4.12

    @staticmethod
    def q_dt(dt):
        return int(round((dt/np.pi)*(1 << 24)))         # DTQ

    @staticmethod
    def q_dw(x):
        return int(round(x*32768)) & 0xFFFF             # Q1.15 signed

    @staticmethod
    def q_namp(D, dt):
        return int(round(np.sqrt(6.0*D*dt)*LSB_PER_RAD))

    # ---------- configuration ----------
    def load_graph(self, A):
        """J = -A, packed 8 x signed-4-bit per word, flat index i*16 + j."""
        A = np.asarray(A, float)
        n = A.shape[0]
        self.n = n
        J = np.round(-A).astype(int)
        assert J.min() >= -8 and J.max() <= 7, "J must fit signed 4-bit"
        nib = np.zeros(16*16, dtype=int)
        for i in range(n):
            for j in range(n):
                nib[i*16 + j] = J[i, j] & 0xF
        for k in range(16*16//8):
            self.wr(self.JPACK + 4*k,
                    sum(int(nib[k*8 + b]) << (4*b) for b in range(8)))
        self.wr(self.NSPINS, n)

    def configure(self, dt, Kc=1.0, Ks=0.6, D=0.0, seed=1, max_steps=200000,
                  tol=40, stride=64, hold=8, anneal=None, dw=None, w=None):
        self.wr(self.KC, self.q_kc(Kc))
        self.wr(self.DTQ, self.q_dt(dt))
        self.wr(self.NAMP, self.q_namp(D, dt))
        self.wr(self.SEED, seed)
        self.wr(self.MAXSTEPS, max_steps)
        self.wr(self.TOL, tol)
        self.wr(self.STRIDE, stride)
        self.wr(self.HOLD, hold)
        if anneal is None:
            self.wr(self.KS, self.q_kc(Ks))
            self.wr(self.KS_INC, 0)
        else:
            ks0, ks1, nstep = anneal
            self.wr(self.KS_START, self.q_kc(ks0))
            self.wr(self.KS_STOP, self.q_kc(ks1))
            self.wr(self.KS_INC,
                    int(((self.q_kc(ks1) - self.q_kc(ks0)) << 16)//max(nstep, 1)))
        for i in range(16):
            self.wr(self.W + 4*i, 4096 if w is None or i >= len(w) else self.q_kc(w[i]))
            self.wr(self.DW + 4*i, 0 if dw is None or i >= len(dw) else self.q_dw(dw[i]))
        self.reset()                     # LFSR <- SEED; clears engine phase, so set_phase after
        self.dt = dt

    def set_phase(self, phi_rad):
        for i, p in enumerate(np.atleast_1d(phi_rad)):
            self.wr(self.PHASE + 4*i, int(round(p*LSB_PER_RAD)) & 0xFFFF)
        self.wr(self.CTRL, self.B_LOADPH)

    def set_freq(self, mult=None, fw0=256):
        """mult[i] = 2 makes oscillator i run at 2f (frequency multiplication)."""
        self.wr(self.FW0, fw0)
        for i in range(16):
            k = 1 if mult is None or i >= len(mult) else int(mult[i])
            self.wr(self.FW + 4*i, fw0*k)
        self.fw0 = fw0
        self.fw = [fw0*(1 if mult is None or i >= len(mult) else int(mult[i]))
                   for i in range(16)]

    def set_fw(self, fw, fw0=256):
        """Raw frequency words, e.g. FW[1] = 260 for a 1.6 % drift. The SHIL
        reference of the v2 readout runs at 2*FW0."""
        self.wr(self.FW0, fw0)
        self.fw0 = fw0
        self.fw = [int(fw[i]) if i < len(fw) else fw0 for i in range(16)]
        for i in range(16):
            self.wr(self.FW + 4*i, self.fw[i])

    def set_phtap(self, deg=40.0):
        """Kim phase-ID tap spacing. 40 deg reproduces the 0/2/4/6
        inverter-stage taps of a 9-stage ROSC. 0x1C71 is the reset default."""
        self.phtap_q = int(round(deg*(1 << PW)/360.0)) & 0xFFFF
        self.wr(self.PHTAP, self.phtap_q)

    # ---------- run ----------
    def solve(self, timeout_s=10.0):
        import time
        self.wr(self.CTRL, self.B_START)
        t0 = time.time()
        while not (self.rd(self.STATUS) & 0x2):
            if time.time() - t0 > timeout_s:
                raise TimeoutError("engine did not finish")
        st = self.rd(self.STATUS)
        return dict(settled=bool(st & 0x1), steps=self.rd(self.STEPCNT),
                    phase=self.phases())

    def phases(self, deg=True):
        p = np.array([self.rd(self.PHASE + 4*i) for i in range(self.n)])
        return p*DEG_PER_LSB if deg else p*TWO_PI/(1 << PW)

    def phase_ids(self):
        """Kim 4-bit thermometer codes, all latched on the SAME base edge.

        NOTE: this is phase RELATIVE to the base oscillator's zero crossing,
        not absolute phase.
        """
        return np.array([self.rd(self.PHASEID + 4*i) & 0xF
                         for i in range(self.n)])

    def readout(self, window=200, duty=0.5, base=0, timeout_s=10.0):
        import time
        self.wr(self.DUTY, int(round(duty*(1 << PW))) & 0xFFFF)
        self.wr(self.WINDOW, window)
        self.wr(self.BASE, base)
        self.wr(self.CTRL, self.B_RDSTART)
        t0 = time.time()
        while not (self.rd(self.STATUS) & 0x8):
            if time.time() - t0 > timeout_s:
                raise TimeoutError("readout did not finish")
        tot = self.rd(self.TOTAL)
        xc = np.array([self.rd(self.XORC + 4*i) for i in range(self.n)])
        ec = np.array([self.rd(self.EDGEC + 4*i) for i in range(self.n)])
        tg = np.array([self.rd(self.TOGC + 4*i) for i in range(self.n)])
        dq = self.rd(self.DFFQ)
        # ---- v2 readout: RS-FF, quadrature D-FF, SHIL-edge D-FF ----
        rs = np.array([self.rd(self.RSC + 4*i) for i in range(self.n)])
        t2 = np.array([self.rd(self.TOG2C + 4*i) for i in range(self.n)])
        ts = np.array([self.rd(self.TOGSC + 4*i) for i in range(self.n)])
        pid = np.array([self.rd(self.PHASEID + 4*i) & 0xF for i in range(self.n)])
        d2, ds = self.rd(self.DFF2Q), self.rd(self.DFFSQ)
        bits = lambda q: np.array([(q >> i) & 1 for i in range(self.n)])
        self.duty_q, self.window, self.base = (int(round(duty*(1 << PW))) & 0xFFFF,
                                               window, base)
        return dict(C=100.0*xc/tot, edge=ec, ratio=ec/max(ec[base], 1),
                    toggle=tg/max(window, 1),
                    dff=bits(dq),
                    total=tot,
                    D=100.0*rs/tot,                     # RS-FF, duty-immune
                    toggle2=t2/max(window, 1),          # quadrature D-FF
                    toggle_min=np.minimum(tg, t2)/max(window, 1),
                    togS=ts/max(2*window, 1),           # ~1 for 1x, 0 for locked 2f
                    dff2=bits(d2), dffs=bits(ds),
                    raw=dict(total=tot, xor=xc, edge=ec, tog=tg, rs=rs, tog2=t2,
                             togs=ts, pid=pid, dff=bits(dq), dff2=bits(d2),
                             dffs=bits(ds)))

    # ---------- v2: the readout twin ----------
    def twin(self):
        """oim_sim.readout_twin fed with exactly what the hardware holds."""
        import oim_sim as m
        ph = [self.rd(self.PHASE + 4*i) for i in range(16)]
        return m.readout_twin(ph, self.fw, base=self.base, duty_q=self.duty_q,
                              window=self.window, fw0=self.fw0,
                              phtap_q=self.phtap_q)

    def matches_twin(self, r):
        """True when EVERY counter of the last readout equals the twin."""
        t, raw, n = self.twin(), r["raw"], self.n
        return (raw["total"] == t["total"]
                and all(np.array_equal(raw[k], t[k][:n])
                        for k in ("xor", "edge", "tog", "rs", "tog2", "togs",
                                  "pid", "dff", "dff2", "dffs")))


print("driver defined")

## 2. Bit-exactness against the Python twin

**Do not skip this.** The RTL is already proven bit-exact against the twin in
simulation, so if the board disagrees here the design is not wrong - a register
value is. That turns hardware debugging into comparing seven numbers.

In [ ]:
import oim_sim as m

A = m.size6_graph()
Ks = m.KS_DEFAULT            # 0.30, inside the measured window m.KS_WINDOW
dt = m.stable_dt(A, Ks)

eng = OIMEngine()
eng.load_graph(A)
eng.configure(dt=dt, Kc=1.0, Ks=Ks, D=0.0, seed=1,
              max_steps=200, tol=0, stride=0xFFFF, hold=0xFF)  # never settle early
phi0 = m._start(7, A.shape[0])
eng.set_phase(phi0)
eng.solve()
hw = (np.round(np.array(eng.phases(deg=False))*LSB_PER_RAD).astype(int)) & 0xFFFF

cfg, _ = m.make_fixed_cfg(A, Ks=Ks, D=0.0, seed=1)
cfg.HOLD = 10**9
p0q = (np.round(phi0*m.LSB_PER_RAD).astype(np.int64)) & m.PHASE_MASK
sw = m.fixed_run(cfg, p0q, max_steps=200)["phase"]

print("hardware :", [f"{v:04X}" for v in hw])
print("twin     :", [f"{int(v):04X}" for v in sw])
print("BIT-EXACT" if np.array_equal(hw, np.asarray(sw, int)) else "*** MISMATCH ***")

## 3. Solve a graph

In [ ]:
A = m.size6_graph()
n = A.shape[0]
best = m.brute_force_maxcut(A)[0]
dt = m.stable_dt(A, m.KS_DEFAULT)

eng.load_graph(A)
eng.set_freq()
cuts, phases = [], []
for s in range(20):
    eng.configure(dt=dt, Kc=1.0, D=0.004, seed=1234 + s, max_steps=200000,
                  tol=40, stride=64, hold=8, anneal=(0.0, m.KS_DEFAULT, 400))
    eng.set_phase(np.random.default_rng(s).uniform(0, TWO_PI, n))
    eng.solve()
    ph = np.radians(eng.phases())
    cuts.append(m.cut_value(A, m.spins_from_phase(ph)))
    phases.append(ph)
cuts = np.array(cuts)
print(f"20 runs: best {cuts.max():.0f}/{best:.0f}, mean {cuts.mean():.2f}, "
      f"P(optimum) = {np.mean(cuts >= best):.2f}")

k = int(np.argmax(cuts))
rel = np.degrees(np.mod(phases[k] - phases[k][0], TWO_PI))
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(np.arange(1, n + 1), rel,
       color=["tab:green" if x < 90 or x > 270 else "tab:blue" for x in rel])
ax.axhline(180, color="k", ls=":")
ax.set_yticks([0, 90, 180, 270, 360])
ax.set_xlabel("oscillator")
ax.set_ylabel("phase - phase_1 (deg)")
ax.set_title(f"settled phases, cut {cuts.max():.0f}/{best:.0f}")
plt.show()

## 4. The four detectors

Madam's readout table. Oscillator 2 is forced to **2f** so the
frequency-multiplication state appears.

In [ ]:
A5 = m.size5_graph()
eng.load_graph(A5)
eng.configure(dt=0.0, Kc=1.0, Ks=m.KS_DEFAULT, D=0.0, max_steps=1)   # DTQ = 0: latch only
eng.set_phase(np.radians([0.0, 37.0, 180.0, 180.0, 0.0]))
eng.set_freq(mult=[1, 2, 1, 1, 1])          # osc2 at 2f
eng.solve()
r = eng.readout(window=200, duty=0.5, base=0)

M_th = 20.0
print(f"{'osc':>4} {'C(%)':>8} {'f_i/f_b':>9} {'DFF':>5} {'toggle':>8}   state")
S = []
for i in range(A5.shape[0]):
    if abs(r["ratio"][i] - 1) > 0.05:
        st, sp = "frequency multiplication (2f)", 0
    elif r["toggle"][i] > 0.02:
        st, sp = "unlocked / drifting", 0
    elif r["C"][i] < M_th:
        st, sp = "in-phase (0 deg)", +1
    elif r["C"][i] > 100 - M_th:
        st, sp = "anti-phase (180 deg)", -1
    elif abs(r["C"][i] - 50) < M_th:
        st, sp = "quadrature (90 deg)", 0
    else:
        st, sp = "UNREADABLE", 0
    S.append(sp)
    print(f"{i+1:>4} {r['C'][i]:8.2f} {r['ratio'][i]:9.4f} {r['dff'][i]:5d} "
          f"{r['toggle'][i]:8.3f}   {st}")

S = np.array(S, float)
print("\nS =", S.astype(int))
gp, gm, free = m.spins_to_partition(S)
print(f"groups {gp} | {gm}   FREE {free}")
for s in m.expand_free(S):
    p, q = m.partition_of(s)
    print(f"   {p} | {q}   cut {m.cut_value(A5, s):.0f}")

## 4b. Every detector against phase shift - hardware vs the readout twin

The v2 bitstream (ID `0x4F494D02`) adds the three pipeline detectors to the
readout: the **RS flip-flop** (SET on the base edge, RESET on the V_i edge),
a second **D-FF on the quadrature base edge** (base + 90 deg), and a **D-FF on
the SHIL edge** (2 f0). Oscillator 2 is swept through 360 deg against the base.

Every tenth point is compared with `oim_sim.readout_twin`, counter for counter.
Like section 2, a mismatch means a register value is wrong, not the design.

In [ ]:
assert eng.version >= 2, "sections 4b-4g need the v2 bitstream (ID 0x4F494D02)"
A5 = m.size5_graph()
eng.load_graph(A5)
eng.configure(dt=0.0, Kc=1.0, Ks=0.0, D=0.0, max_steps=1)   # DTQ = 0: latch only
eng.set_phtap(40.0)


def det_sweep(n_pts=181, duty=0.5, window=200, fw1=None, osc=1, check_every=10):
    """Oscillator `osc` swept 0..360 deg against the base (osc 0 at 0 deg)."""
    eng.set_fw([256]*16 if fw1 is None else [256, fw1] + [256]*14)
    deltas = np.linspace(0, 360, n_pts)
    keys = ("C", "D", "dff", "dff2", "ratio", "togS")
    out = {k: [] for k in keys}
    checked = exact = 0
    for k, d in enumerate(deltas):
        ph = np.zeros(eng.n)
        ph[osc] = np.radians(d)
        eng.set_phase(ph)
        eng.solve()
        rr = eng.readout(window=window, duty=duty, base=0)
        for key in keys:
            out[key].append(rr[key][osc])
        if check_every and k % check_every == 0:
            checked += 1
            exact += bool(eng.matches_twin(rr))
    return deltas, {k: np.array(v) for k, v in out.items()}, (exact, checked)


d, R, (ok, nchk) = det_sweep()
print(f"hardware vs readout twin: {ok}/{nchk} points BIT-EXACT on every counter"
      + ("" if ok == nchk else "   *** MISMATCH -- compare FW, FW0, DUTY, PHTAP ***"))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(d, R["C"], lw=1.6, label="XOR  C")
ax[0].plot(d, R["D"], lw=1.6, label="RS flip-flop  D")
ax[0].set_xticks([0, 90, 180, 270, 360])
ax[0].set_xlabel("phase shift (deg)")
ax[0].set_ylabel("detector output (%)")
ax[0].set_title("XOR folds at 180 deg; the RS sawtooth does not")
ax[0].legend(fontsize=7)
ax[1].step(d, R["dff"] + 1.5, where="mid", lw=1.6, label="D-FF on base edge")
ax[1].step(d, R["dff2"], where="mid", lw=1.6, label="D-FF on base + 90 deg")
ax[1].set_yticks([0, 1, 1.5, 2.5])
ax[1].set_yticklabels(["0", "1", "0", "1"])
ax[1].set_xticks([0, 90, 180, 270, 360])
ax[1].set_xlabel("phase shift (deg)")
ax[1].set_title("the two D-FFs: a 2-bit quadrant code")
ax[1].legend(fontsize=7, loc="center right")
plt.show()

## 4c. Duty immunity

The XOR's anti-phase peak clips to 200 d as the detector duty falls, which is
what produces the duty cliff. The RS flip-flop only uses rising edges, and a
rising edge does not move when the duty changes.

With FW = 256 the waveform has 256 samples per period, so the duty is
quantised to that grid: 0.45 is really 116/256, and the XOR peak reads
2 x 116/256 = 90.63 % rather than 90.00 %.

In [ ]:
d, R50, _ = det_sweep(n_pts=91, duty=0.50, check_every=0)
_, R40, _ = det_sweep(n_pts=91, duty=0.40, check_every=0)

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(d, R50["C"], color="tab:blue", lw=1.6, label="XOR, duty 0.5")
ax.plot(d, R40["C"], color="tab:blue", lw=1.2, ls="--", label="XOR, duty 0.4")
ax.plot(d, R50["D"], color="tab:red", lw=1.6, label="RS-FF, duty 0.5")
ax.plot(d, R40["D"], color="tab:red", lw=1.2, ls="--", label="RS-FF, duty 0.4 (on top of 0.5)")
ax.set_xticks([0, 90, 180, 270, 360])
ax.set_xlabel("phase shift (deg)")
ax.set_ylabel("detector output (%)")
ax.set_title("XOR moves with duty; the RS flip-flop does not")
ax.legend(fontsize=7)
plt.show()

eng.set_fw([256]*16)
eng.set_phase(np.radians([0.0, 180.0, 0.0, 0.0, 0.0]))
eng.solve()
print(" duty   XOR C at 180 deg   RS-FF D at 180 deg")
for du in (0.50, 0.45, 0.40, 0.35, 0.30):
    rr = eng.readout(window=200, duty=du, base=0)
    print(f" {du:.2f}   {rr['C'][1]:14.2f}   {rr['D'][1]:16.2f}")

## 4d. Lock: the edge counter, one D-FF, two D-FFs

Oscillator 2 is detuned by a few percent. The edge ratio barely moves, so a
ratio test waves it through; the D-FF toggle rate sees the phase sliding past
the sampling instant. The pipeline takes `min(toggle, toggle2)`: with real
jitter a single D-FF false-trips near 0 and 180 deg, two cannot both.

The PL readout has **no jitter source**, so the right-hand panel is zero for
both flip-flops at every offset. The jitter study itself lives in
`oim_pipeline.py`; this panel shows the board adds no false trips of its own.

In [ ]:
W = 400                                      # the pipeline's C2 window
TOGGLE_TOL, RATIO_TOL = 0.01, 0.25
fws = [256, 257, 258, 259, 260, 262, 264, 268, 272]
drift, ratio, t1, t2 = [], [], [], []
for fw1 in fws:
    eng.set_fw([256, fw1])
    eng.set_phase(np.radians([0.0, 90.0, 0.0, 0.0, 0.0]))
    eng.solve()
    rr = eng.readout(window=W, duty=0.5, base=0)
    drift.append(100.0*(fw1/256.0 - 1.0))
    ratio.append(rr["ratio"][1])
    t1.append(rr["toggle"][1])
    t2.append(rr["toggle2"][1])
t1, t2 = np.array(t1), np.array(t2)

eng.set_fw([256]*16)
offs = np.arange(0, 181, 5)
lk1, lk2 = [], []
for o in offs:
    eng.set_phase(np.radians([0.0, o, 0.0, 0.0, 0.0]))
    eng.solve()
    rr = eng.readout(window=W, duty=0.5, base=0)
    lk1.append(rr["toggle"][1])
    lk2.append(rr["toggle2"][1])

fig, ax = plt.subplots(1, 3, figsize=(11.5, 3.3))
ax[0].axhspan(1 - RATIO_TOL, 1 + RATIO_TOL, color="tab:blue", alpha=.12, lw=0)
ax[0].plot(drift, ratio, "o-", ms=4)
ax[0].set_ylim(0.5, 1.5)
ax[0].set_xlabel("relative drift (%)")
ax[0].set_ylabel("edge ratio f_i/f_base")
ax[0].set_title("edge counter: '1x' at every drift")
ax[1].plot(drift, t1, "o-", ms=4, label="1 D-FF (base edge)")
ax[1].plot(drift, t2, "s-", ms=4, label="D-FF on base + 90 deg")
ax[1].plot(drift, np.minimum(t1, t2), "k--", lw=1, label="min of the two")
ax[1].axhline(TOGGLE_TOL, color="r", ls=":", lw=1, label="toggle_tol 0.01")
ax[1].set_xlabel("relative drift (%)")
ax[1].set_ylabel("toggle rate")
ax[1].set_title("the toggle catches the drift")
ax[1].legend(fontsize=6.5)
ax[2].plot(offs, lk1, "o-", ms=3, label="1 D-FF")
ax[2].plot(offs, lk2, "s-", ms=3, label="D-FF + 90 deg")
ax[2].axhline(TOGGLE_TOL, color="r", ls=":", lw=1)
ax[2].set_ylim(-0.005, 0.05)
ax[2].set_xlabel("locked phase offset (deg)")
ax[2].set_title("locked: no false trips (no jitter on the PL)")
ax[2].legend(fontsize=6.5)
plt.show()

print(" drift %   ratio    tog     tog2    min")
for a, b, c, e in zip(drift, ratio, t1, t2):
    print(f" {a:6.2f}  {b:6.4f}  {c:6.4f}  {e:6.4f}  {min(c, e):6.4f}")

## 4e. The SHIL-edge D-FF

Sampling on the 2 f0 SHIL edge separates the branches by itself: a 1x
oscillator is sampled twice per period at opposite half-cycles, so it toggles
on every sample (about 2 x WINDOW); a 2f oscillator locked to the SHIL is
sampled at the same point of its cycle every time, so it never toggles. A 2f
oscillator that is slightly off 2f toggles a little - the lock test the 2f
branch needs.

In [ ]:
A6 = m.size6_graph()
eng.load_graph(A6)
eng.configure(dt=0.0, Kc=1.0, Ks=0.0, D=0.0, max_steps=1)   # DTQ = 0: latch only
eng.set_freq(mult=[1, 1, 1, 1, 2, 1])
phs, mults = [0, 90, 180, 270, 37, 45], [1, 1, 1, 1, 2, 1]
eng.set_phase(np.radians(phs))
eng.solve()
rr = eng.readout(window=200, duty=0.5, base=0)
raw = rr["raw"]
print(" osc  phase  f |     C       D  | edges  tog  tog2  togS")
for i in range(6):
    print(f" {i:3d}  {phs[i]:5d}  {mults[i]} | {rr['C'][i]:6.2f}  {rr['D'][i]:6.2f} | "
          f"{raw['edge'][i]:5d}  {raw['tog'][i]:3d}  {raw['tog2'][i]:4d}  {raw['togs'][i]:4d}")
print("hardware vs readout twin:", "BIT-EXACT" if eng.matches_twin(rr) else "*** MISMATCH ***")

fw4s = np.arange(500, 525, 2)
togs, rat = [], []
for fw4 in fw4s:
    eng.set_fw([256, 256, 256, 256, int(fw4), 256])
    eng.set_phase(np.radians(phs))
    eng.solve()
    rr = eng.readout(window=200, duty=0.5, base=0)
    togs.append(rr["raw"]["togs"][4])
    rat.append(rr["ratio"][4])
off = 100.0*(fw4s/512.0 - 1.0)

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.2))
ax[0].plot(off, togs, "o-", ms=4, color="tab:red", label="2f oscillator")
ax[0].axhline(2*200 - 1, color="tab:blue", ls="--", lw=1, label="a 1x oscillator (2W - 1)")
ax[0].set_xlabel("offset from 2 f0 (%)")
ax[0].set_ylabel("SHIL-edge toggles (W = 200)")
ax[0].set_title("togS = 0 only when locked at 2f")
ax[0].legend(fontsize=7)
ax[1].plot(off, rat, "o-", ms=4)
ax[1].set_ylim(1.5, 2.5)
ax[1].set_xlabel("offset from 2 f0 (%)")
ax[1].set_ylabel("edge ratio")
ax[1].set_title("the edge counter only says '2x'")
plt.show()

## 4f. Why frequency must be checked first

A 2f oscillator has no phase relative to the base, but every phase detector
still returns a number for it. The XOR gives a flat 50 %; the RS flip-flop
gives values that look like ordinary phase readings. If the frequency check
does not come first, the pipeline reports a spin for an oscillator that has
none.

In [ ]:
eng.load_graph(A5)
eng.configure(dt=0.0, Kc=1.0, Ks=0.0, D=0.0, max_steps=1)   # DTQ = 0: latch only
eng.set_freq(mult=[1, 2, 1, 1, 1])
print(" 2f oscillator   XOR C    RS-FF D   RS-FF decodes to   edge ratio")
for th in (0, 37, 210):
    eng.set_phase(np.radians([0.0, th, 0.0, 0.0, 0.0]))
    eng.solve()
    rr = eng.readout(window=200, duty=0.5, base=0)
    dec = 360.0*(1.0 - rr["D"][1]/100.0)
    print(f"   phase {th:3d}     {rr['C'][1]:6.2f}   {rr['D'][1]:7.2f}   {dec:9.1f} deg"
          f"        {rr['ratio'][1]:.4f}")
eng.set_freq()

## 4g. The operating distribution: every detector on settled phases

Margins must be judged on the phases the machine actually produces, not on a
uniform sweep. Forty solves of the size-6 graph - noisy exploration, then
noise off until the settling test passes - and one readout each.

In [ ]:
A6 = m.size6_graph()
eng.load_graph(A6)
eng.set_freq()
dt6 = m.stable_dt(A6, m.KS_DEFAULT)
rng = np.random.default_rng(11)
rel_all, C_all, D_all = [], [], []
for s in range(40):
    eng.configure(dt=dt6, Kc=1.0, Ks=m.KS_DEFAULT, D=0.004, seed=500 + s,
                  max_steps=3000, tol=0)                      # noisy exploration
    eng.set_phase(rng.uniform(0, TWO_PI, 6))
    eng.solve()
    ph = np.radians(eng.phases())
    eng.configure(dt=dt6, Kc=1.0, Ks=m.KS_DEFAULT, D=0.0,
                  max_steps=200000, tol=40, stride=64, hold=8)  # noise off: settle
    eng.set_phase(ph)
    eng.solve()
    rr = eng.readout(window=200, duty=0.5, base=0)
    p = eng.phases()
    rel = np.mod(p - p[0], 360.0)
    rel_all += list(rel[1:])
    C_all += list(rr["C"][1:])
    D_all += list(rr["D"][1:])
rel_all = np.array(rel_all)
d0 = np.abs(((rel_all + 180) % 360) - 180)          # distance to 0 deg
frac = np.mean(np.minimum(d0, np.abs(rel_all - 180)) < 15.0)

x = np.linspace(0, 360, 721)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.3))
ax[0].hist(rel_all, bins=np.arange(0, 365, 5), color="tab:green")
ax[0].set_xticks([0, 90, 180, 270, 360])
ax[0].set_xlabel("settled phase - phase_1 (deg)")
ax[0].set_ylabel("count")
ax[0].set_title(f"{100*frac:.1f} % within 15 deg of 0 / 180")
ax[1].plot(rel_all, C_all, "o", ms=3, label="XOR C")
ax[1].plot(rel_all, D_all, "s", ms=3, label="RS-FF D")
ax[1].set_xticks([0, 90, 180, 270, 360])
ax[1].set_xlabel("settled phase - phase_1 (deg)")
ax[1].set_ylabel("detector output (%)")
ax[1].set_title("where the settled states land on each detector")
ax[1].legend(fontsize=7)
plt.show()

eng.load_graph(m.size5_graph())     # section 5 expects the size-5 graph


## 5. The unreadable band, measured on hardware

Madam's core experiment. The engine is left idle (`max_steps = 1` just latches
the phases you write); only the waveform generator and the counters run.

In [ ]:
eng.configure(dt=0.0, Kc=1.0, Ks=0.0, D=0.0, max_steps=1)   # DTQ = 0: latch only


def hw_sweep(n_pts=181, duty=0.5, window=200, mult=None):
    eng.set_freq(mult=mult)
    deltas = np.linspace(0, 360, n_pts)
    C, ratio = [], []
    for d in deltas:
        eng.set_phase(np.radians([0.0, d, 0.0, 0.0, 0.0]))
        eng.solve()
        rr = eng.readout(window=window, duty=duty, base=0)
        C.append(rr["C"][1])
        ratio.append(rr["ratio"][1])
    return deltas, np.array(C), np.array(ratio)


M_th = 20.0
d, C, _ = hw_sweep(n_pts=181, duty=0.500)
_, C47, _ = hw_sweep(n_pts=181, duty=0.472)

cls = m.classify_C(C, M_th)
bands = m.bands_from_classes(d, cls)
col = {m.CLASS_PLUS: "tab:green", m.CLASS_MINUS: "tab:blue",
       m.CLASS_FREE: "tab:orange", m.CLASS_BAD: "tab:red"}
name = {m.CLASS_PLUS: "+1", m.CLASS_MINUS: "-1",
        m.CLASS_FREE: "free (0)", m.CLASS_BAD: "UNREADABLE"}

fig, ax = plt.subplots(figsize=(8, 3.6))
for c in col:
    for lo, hi in bands[c]:
        ax.axvspan(lo, hi, color=col[c], alpha=.18, lw=0)
ax.plot(d, C, "k-", lw=1.6, label="duty 0.500")
ax.plot(d, C47, "m--", lw=1.2, label="duty 0.472")
for y in (M_th, 50 - M_th, 50 + M_th, 100 - M_th):
    ax.axhline(y, color="k", lw=.5, ls=":")
ax.set_xticks([0, 90, 180, 270, 360])
ax.set_xlabel("phase shift (deg)")
ax.set_ylabel("C (%)")
step = d[1] - d[0]
bad = float(np.sum(cls == m.CLASS_BAD)*step)
ax.set_title(f"hardware XOR readout, M_th = {M_th:.0f} % : UNREADABLE = {bad:.0f} deg of 360")
ax.legend(fontsize=7, loc="lower center")
plt.show()

for c in (m.CLASS_PLUS, m.CLASS_FREE, m.CLASS_MINUS, m.CLASS_BAD):
    print(f"  {name[c]:11s}", [(round(a, 1), round(z, 1)) for a, z in bands[c]])
print(f"\n  measured UNREADABLE = {bad:.1f} deg")
print(f"  closed form          = "
      f"{m.unreadable_bands_analytic(M_th, 0.5)['bad_width']:.1f} deg")

## 6. Degeneracy

Left: the XOR reads 50 % at *every* phase for a 2f oscillator, so it cannot
separate 2f from a 90 deg lock. Right: the edge counter can.

In [ ]:
d1, C1, R1 = hw_sweep(n_pts=91, mult=[1, 1, 1, 1, 1])
d2, C2, R2 = hw_sweep(n_pts=91, mult=[1, 2, 1, 1, 1])

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.2))
ax[0].plot(d1, C1, lw=1.6, label="f_i = f0")
ax[0].plot(d2, C2, lw=1.6, label="f_i = 2 f0")
ax[0].axhline(50, color="k", ls=":", lw=.8)
ax[0].set_xticks([0, 90, 180, 270, 360])
ax[0].set_xlabel("phase shift (deg)")
ax[0].set_ylabel("C (%)")
ax[0].set_title("XOR cannot separate them")
ax[0].legend(fontsize=7)
ax[1].plot(d1, R1, lw=1.6, label="f_i = f0")
ax[1].plot(d2, R2, lw=1.6, label="f_i = 2 f0")
ax[1].set_ylim(0.5, 2.5)
ax[1].set_xticks([0, 90, 180, 270, 360])
ax[1].set_xlabel("phase shift (deg)")
ax[1].set_ylabel("f_i / f_base")
ax[1].set_title("the edge counter can")
ax[1].legend(fontsize=7)
plt.show()

### Frequency tuning recovers all degenerate optima (K4)

Sweep the per-oscillator detuning DW - the digital stand-in for the paper's
tuning capacitors C1 and C2.

In [ ]:
A4 = m.k4_graph()
best4, _, deg4 = m.brute_force_maxcut(A4)
dt4 = m.stable_dt(A4, m.KS_DEFAULT)
rng = np.random.default_rng(3)
mism = rng.standard_normal(4)
mism -= mism.mean()
mism /= np.abs(mism).max()

tunes = np.linspace(-m.KS_TUNE_MAX, m.KS_TUNE_MAX, 29)
found, cuts4 = [], []
eng.load_graph(A4)
eng.set_freq()
for tu in tunes:
    eng.configure(dt=dt4, Kc=1.0, D=0.0, seed=3, max_steps=200000,
                  tol=40, stride=64, hold=8, anneal=(0.0, m.KS_DEFAULT, 400),
                  dw=tu*mism)
    eng.set_phase(m._start(3, 4))
    eng.solve()
    s = m.spins_from_phase(np.radians(eng.phases()))
    s = s*s[0]
    cuts4.append(m.cut_value(A4, s))
    found.append(tuple(int(v) for v in s))

uniq = {}
for tu, k, c in zip(tunes, found, cuts4):
    if c >= best4:
        uniq.setdefault(k, []).append(tu)
print(f"K4: recovered {len(uniq)} of {deg4} degenerate optima")

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.2))
ax[0].plot(tunes, cuts4, "o-", ms=4)
ax[0].axhline(best4, color="r", ls="--", lw=.8)
ax[0].set_xlabel("detuning scale")
ax[0].set_ylabel("cut")
ax[0].set_title("cut vs frequency tuning")
for idx, (k, tl) in enumerate(uniq.items()):
    p, q = m.partition_of(np.array(k, float))
    ax[1].plot(tl, [idx]*len(tl), "o", ms=6, label=f"{p} | {q}")
ax[1].set_xlabel("detuning scale")
ax[1].set_ylabel("optimum index")
ax[1].set_title(f"{len(uniq)} of {deg4} optima")
ax[1].legend(fontsize=7)
plt.show()

## 7. Ks / Kc sweep - measuring the useful window

Each run anneals WITH noise for T = 60, then relaxes with D = 0 for T = 40, so
the phases are on true fixed points when they are measured (the same method as
`oim_sim.ks_ratio_sweep`). Binarised = the fraction of spins within 5 deg of
0 or 180.

The window is **measured, not assumed**: it is the longest run of Ks/Kc where
P(optimum) >= 0.95 **and** binarised >= 0.95. On the size-6 graph a 0.025-step
sweep gives **0.05 to 0.45** - wider than the 0.1 to 0.4 quoted earlier, on
both sides. That is `m.KS_WINDOW`; KS_DEFAULT = 0.30 sits inside it and above
the frequency-tuning range (0.25).

In [ ]:
T_ANNEAL, T_RELAX = 60.0, 40.0            # same schedule as oim_sim.ks_ratio_sweep


def ks_sweep(A, ratios, R=20, D=0.004):
    """P(hit optimum) and fraction of spins binarised, per Ks/Kc, on hardware."""
    A = np.asarray(A, float)
    n = A.shape[0]
    best = m.brute_force_maxcut(A)[0]
    eng.load_graph(A)
    eng.set_freq()
    popt, binar = [], []
    for Ks in ratios:
        dt = m.stable_dt(A, max(Ks, 0.1))
        ok = bn = 0.0
        for s in range(R):
            # 1. anneal WITH noise at fixed Ks (tol = 0: run the full schedule)
            eng.configure(dt=dt, Kc=1.0, Ks=Ks, D=D, seed=900 + s,
                          max_steps=int(round(T_ANNEAL/dt)), tol=0)
            eng.set_phase(np.random.default_rng(s).uniform(0, TWO_PI, n))
            eng.solve()
            ph = np.radians(eng.phases())
            # 2. relax with D = 0, so the phases sit on true fixed points when read
            eng.configure(dt=dt, Kc=1.0, Ks=Ks, D=0.0,
                          max_steps=int(round(T_RELAX/dt)), tol=0)
            eng.set_phase(ph)
            eng.solve()
            rel = np.mod(eng.phases() - eng.phases()[0], 360.0)
            off = np.minimum(rel % 180, 180 - (rel % 180))
            ok += m.cut_value(A, m.spins_from_phase(np.radians(eng.phases()))) >= best
            bn += np.mean(off < 5.0)            # fraction of SPINS binarised
        popt.append(ok/R)
        binar.append(bn/R)
    return np.array(popt), np.array(binar)


def ks_window(ratios, popt, binar, p_min=0.95, b_min=0.95):
    """Longest contiguous run where both criteria hold -> (lo, hi) or None."""
    good = (popt >= p_min) & (binar >= b_min)
    best_run, run = (), []
    for k, g_ok in enumerate(good):
        run = run + [k] if g_ok else []
        if len(run) > len(best_run):
            best_run = tuple(run)
    return (ratios[best_run[0]], ratios[best_run[-1]]) if best_run else None


A = m.size6_graph()
ratios = np.round(np.arange(0.0, 0.601, 0.025), 3)
popt, binar = ks_sweep(A, ratios, R=40)
win = ks_window(ratios, popt, binar)

fig, ax = plt.subplots(figsize=(7, 3.3))
if win is not None:
    ax.axvspan(win[0], win[1], color="tab:blue", alpha=.10, lw=0)
ax.plot(ratios, popt, "s-", color="tab:orange", ms=4, label="P(hit optimum)")
ax.plot(ratios, binar, "^-", color="tab:green", ms=4, label="fraction of spins binarised")
ax.axvline(m.KS_DEFAULT, color="k", ls="--", lw=.8, label=f"KS_DEFAULT = {m.KS_DEFAULT}")
ax.set_xlabel("Ks / Kc")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=7, loc="lower left")
ax.set_title("measured window (shaded): " +
             (f"Ks/Kc {win[0]:.3f} to {win[1]:.3f}" if win else "none"))
plt.show()

print(" Ks/Kc   P(opt)  binarised")
for a, p, b in zip(ratios, popt, binar):
    print(f" {a:5.3f}   {p:6.3f}   {b:6.3f}")
print("\nmeasured window: " + (f"{win[0]:.3f} - {win[1]:.3f}" if win else "none")
      + f"    m.KS_WINDOW: {m.KS_WINDOW[0]:.2f} - {m.KS_WINDOW[1]:.2f}")

### 7b. The window depends on the graph

Same measurement on four more graphs (20 runs, Ks/Kc in steps of 0.05). On
frustrated graphs a weak Ks has genuinely **non-binary** fixed points: the cut
still rounds to the optimum, but the phases do not sit at 0/180, which is what
the readout needs. So the binarisation edge moves up with the graph, and a
single fixed Ks cannot serve every instance; ramping Ks up to a value above the
binarisation edge (the `anneal=` option of `configure`) is the remedy.

In [ ]:
graphs = [("size-5", m.size5_graph()), ("K4", m.k4_graph()),
          ("random-10", m.random_graph(10, 20, seed=3)),
          ("surrogate-16", m.surrogate16_graph())]
r7b = np.round(np.arange(0.0, 0.801, 0.05), 3)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.3))
print(f" {'graph':13s} window (P >= 0.95 and binarised >= 0.95)")
for name, Ag in graphs:
    p, b = ks_sweep(Ag, r7b, R=20)
    w = ks_window(r7b, p, b)
    ln, = ax[0].plot(r7b, p, "o-", ms=3, label=name)
    ax[1].plot(r7b, b, "o-", ms=3, color=ln.get_color(), label=name)
    print(f" {name:13s} " + (f"{w[0]:.2f} - {w[1]:.2f}" if w else "none"))
for a_ in ax:
    a_.axvspan(*m.KS_WINDOW, color="tab:blue", alpha=.08, lw=0)
    a_.set_xlabel("Ks / Kc")
    a_.set_ylim(0, 1.05)
    a_.legend(fontsize=7)
ax[0].set_ylabel("P(hit optimum)")
ax[1].set_ylabel("fraction of spins binarised")
ax[0].set_title("the cut is found across a wide range")
ax[1].set_title("binarisation needs Ks above a graph-dependent edge")
plt.show()
print("shaded: the size-6 window m.KS_WINDOW =", m.KS_WINDOW)

### 7c. Use the right Ks for the graph you are solving

`KS_DEFAULT = 0.30` was derived on the **size-6** graph. Your own board data
shows it is outside the window for size-5, random-10 and surrogate-16 - the cut
still rounds out correctly there, but the phases do not binarise, so the readout
is left with unresolved spins. `recommend_ks(A)` returns the centre of the
window for the graph you are actually solving.


In [ ]:
for name, Ag in [("size-6", m.size6_graph()), ("K4", m.k4_graph()),
                 ("size-5", m.size5_graph()), ("surrogate-16", m.surrogate16_graph())]:
    ok, win = m.check_ks(Ag, m.KS_DEFAULT)
    ks = m.recommend_ks(Ag, quiet=True)
    flag = "ok" if ok else "*** 0.30 is OUTSIDE this window ***"
    print(f" {name:13s} window {win[0]:.2f}-{win[1]:.2f}   use Ks/Kc = {ks:<6.3f} {flag}")

# solve surrogate-16 on the board at the RIGHT Ks and count unresolved spins
A16 = m.surrogate16_graph(); best16 = m.brute_force_maxcut(A16)[0]
for ks in (m.KS_DEFAULT, m.recommend_ks(A16, quiet=True)):
    eng.load_graph(A16)
    eng.configure(dt=m.stable_dt(A16, ks), Kc=1.0, Ks=ks, D=0.004, seed=7,
                  max_steps=200000, tol=40, stride=64, hold=8)
    eng.set_freq(); eng.set_phase(m._start(7, 16)); eng.solve()
    rel = np.mod(eng.phases() - eng.phases()[0], 360.0)
    off = np.minimum(rel % 180, 180 - (rel % 180))
    cut = m.cut_value(A16, m.spins_from_phase(np.radians(eng.phases())))
    print(f" Ks/Kc = {ks:<6.3f} cut {cut:.0f}/{best16:.0f}   "
          f"worst off-binary {off.max():6.1f} deg   "
          f"binarised {100*np.mean(off < 5):.0f} %")


## 8. The Chris Kim readout

Moy, Ahmed, Chiu, Moy, Sapatnekar and Kim, *A 1,968-node coupled ring oscillator
circuit for combinatorial optimization problem solving*, Nature Electronics **5**,
310-317 (2022).

Each oscillator is tapped at 0, 1, 2 and 3 x PHTAP of phase and all of them are
latched on the **same** base edge, giving a 4-bit thermometer code. Eight legal
codes, about 45 deg resolution, but zero measurement skew between oscillators.

In [ ]:
eng.load_graph(A5)
eng.configure(dt=0.0, Kc=1.0, Ks=m.KS_DEFAULT, D=0.0, max_steps=1)   # DTQ = 0: latch only
eng.set_phtap(40.0)
eng.set_freq()

deltas = np.linspace(0, 350, 36)
codes = []
for d in deltas:
    eng.set_phase(np.radians([0.0, d, 0.0, 0.0, 0.0]))
    eng.solve()
    eng.readout(window=8, duty=0.5, base=0)
    codes.append(int(eng.phase_ids()[1]))
codes = np.array(codes)

# model cross-check removed: this notebook plots hardware only
print(" delta   hardware code")
for d, a in zip(deltas, codes):
    print(f" {d:5.0f}   {a:04b}")
print("\nlegal codes seen:", sorted(set(format(c, "04b") for c in codes)))

fig, ax = plt.subplots(figsize=(7, 3))
ax.step(deltas, [m.kim_decode(int(c)) for c in codes], where="mid", lw=1.6,
        label="hardware phase ID")
ax.plot(deltas, deltas, "k:", lw=1, label="true")
ax.set_xticks([0, 90, 180, 270, 360])
ax.set_xlabel("phase shift (deg)")
ax.set_ylabel("decoded phase (deg)")
ax.legend(fontsize=7)
ax.set_title("Kim 4-bit phase ID, measured on hardware")
plt.show()